In [1]:
import os, sys
# Add backend dir to python path so we can import from ml
sys.path.insert(0, os.path.abspath('..'))


In [2]:
"""
trainer.py
----------
Automatic retraining engine. Combines the original base dataset with all
expert-confirmed TrainingSample rows accumulated in the database, rebuilds
the feature matrix through the shared feature-engineering pipeline, and
retrains the model(s). New best models are versioned, persisted, recorded
in ModelVersion, and the live Predictor is hot-reloaded.
"""
import os, json, pickle, warnings, datetime
import numpy as np
import pandas as pd

In [3]:
warnings.filterwarnings('ignore')

In [4]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    HAS_XGB = False

In [5]:
from ml.predictor import (FEATURES, CATEGORICAL_COLS,
                                    engineer_dataframe, encode_categoricals)


In [6]:
TARGET_COLS = {'food': 'Target_Food', 'beverage': 'Target_Beverage',
               'medicine': 'Target_Medicine', 'agronomic': 'target'}
SAMPLE_TARGET_ATTR = {'food': 'target_food', 'beverage': 'target_beverage',
                      'medicine': 'target_medicine', 'agronomic': 'target_agronomic'}

In [7]:
def _balance(X, y):
    y = pd.Series(y).reset_index(drop=True)
    X = pd.DataFrame(X).reset_index(drop=True)
    counts = y.value_counts(); maxn = counts.max()
    Xs, ys = [], []
    rng = np.random.RandomState(42)
    for cls in counts.index:
        idx = y[y == cls].index
        Xs.append(X.loc[idx]); ys.append(y.loc[idx])
        if len(idx) < maxn:
            extra = rng.choice(idx, size=maxn - len(idx), replace=True)
            Xs.append(X.loc[extra]); ys.append(y.loc[extra])
    return pd.concat(Xs, ignore_index=True), pd.concat(ys, ignore_index=True).to_numpy()

In [8]:
def _candidates():
    m = {
        'random_forest': (RandomForestClassifier(random_state=42),
                          {'n_estimators': [100], 'max_depth': [10, 15]}),
        'gradient_boosting': (GradientBoostingClassifier(random_state=42),
                             {'n_estimators': [100], 'learning_rate': [0.1], 'max_depth': [3, 5]}),
        'svm': (SVC(random_state=42, probability=True),
                {'C': [1, 10], 'kernel': ['rbf']}),
        'neural_network': (MLPClassifier(max_iter=300, random_state=42),
                          {'hidden_layer_sizes': [(100,), (50, 25)], 'learning_rate_init': [0.01]}),
    }
    if HAS_XGB:
        m['xgboost'] = (XGBClassifier(random_state=42, verbosity=0),
                        {'n_estimators': [100], 'learning_rate': [0.1], 'max_depth': [3, 5]})
    return m

In [9]:
def _samples_to_frame(samples):
    """Convert confirmed TrainingSample rows into a base-dataset-shaped frame."""
    rows = []
    for s in samples:
        f = s.features()
        rows.append({
            'Height_m': f.get('height_m'), 'Crown_Diameter_m': f.get('crown_diameter_m'),
            'Trunk_Diameter_m': f.get('trunk_diameter_m'), 'Altitude_m': f.get('altitude_m'),
            'Geographic_Zone': f.get('geographic_zone'),
            'Tree_Growth_Habitat': f.get('tree_growth_habitat'),
            'Topography': f.get('topography'), 'Soil_Texture': f.get('soil_texture'),
            'Farm_Cultivated': f.get('farm_cultivated'),
            'total_parts_used': f.get('total_parts_used'),
            'taste_attribute': f.get('taste_attribute'), 'medicine_use': f.get('medicine_use'),
            'stem_used': f.get('stem_used'), 'plant_use_count': f.get('plant_use_count'),
            'parts_used_count': f.get('parts_used_count'),
            'total_special_attributes': f.get('total_special_attributes'),
            'Target_Food': s.target_food, 'Target_Beverage': s.target_beverage,
            'Target_Medicine': s.target_medicine, 'target': s.target_agronomic,
        })
    return pd.DataFrame(rows)

In [10]:
def retrain(target, config, db, models_module, predictor):
    """Retrain a single target. Returns a metrics dict or None if skipped."""
    from database.models import TrainingSample, ModelVersion

    base = pd.read_csv(config.BASE_DATASET)
    samples = TrainingSample.query.all()
    if samples:
        extra = _samples_to_frame(samples)
        combined = pd.concat([base, extra], ignore_index=True, sort=False)
    else:
        combined = base

    # Re-fit encoders across the union of categories (base + new field data).
    encoders = {c: LabelEncoder().fit(combined[c].astype(str)) for c in CATEGORICAL_COLS}
    df, alt_min, alt_max = engineer_dataframe(combined)
    df = encode_categoricals(df, encoders)
    X_all = df[FEATURES].copy().fillna(df[FEATURES].median())

    col = TARGET_COLS[target]
    y = pd.to_numeric(df[col], errors='coerce')
    valid = y.notna()
    if valid.sum() < 30 or y[valid].nunique() < 2:
        return None  # not enough labelled data yet

    Xb, yb = _balance(X_all[valid], y[valid].astype(float).to_numpy())
    Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.2,
                                          random_state=42, stratify=yb)
    scaler = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = scaler.transform(Xtr), scaler.transform(Xte)

    n_classes = len(np.unique(yb)); avg = 'weighted' if n_classes > 2 else 'binary'
    evals, fitted = {}, {}
    for name, (est, grid) in _candidates().items():
        gs = GridSearchCV(est, grid, cv=3, n_jobs=-1).fit(Xtr_s, ytr)
        model = gs.best_estimator_
        pred = model.predict(Xte_s); proba = model.predict_proba(Xte_s)
        auc = (roc_auc_score(yte, proba, multi_class='ovr') if n_classes > 2
               else roc_auc_score(yte, proba[:, 1]))
        evals[name] = {'accuracy': float(accuracy_score(yte, pred)),
                       'precision': float(precision_score(yte, pred, average=avg, zero_division=0)),
                       'recall': float(recall_score(yte, pred, average=avg, zero_division=0)),
                       'f1': float(f1_score(yte, pred, average=avg, zero_division=0)),
                       'auc': float(auc)}
        fitted[name] = model
    best = max(evals, key=lambda k: evals[k]['accuracy'])

    # Persist artifacts (shared scalers/encoders updated for ALL targets).
    md = config.MODEL_DIR
    with open(os.path.join(md, f'baobab_{target}_best_model.pkl'), 'wb') as f:
        pickle.dump(fitted[best], f)
    meta = {'target': target, 'best_model': best,
            'best_accuracy': evals[best]['accuracy'],
            'n_classes': int(n_classes), 'model_evaluations': evals,
            'retrained_at': datetime.datetime.utcnow().isoformat()}
    with open(os.path.join(md, f'baobab_{target}_metadata.json'), 'w') as f:
        json.dump(meta, f, indent=2)

    scalers_path = os.path.join(md, 'scalers.pkl')
    scalers = pickle.load(open(scalers_path, 'rb'))
    scalers[target] = scaler
    pickle.dump(scalers, open(scalers_path, 'wb'))
    pickle.dump(encoders, open(os.path.join(md, 'label_encoders.pkl'), 'wb'))
    json.dump({'altitude_min': alt_min, 'altitude_max': alt_max},
              open(os.path.join(md, 'feature_meta.json'), 'w'), indent=2)

    # Version bookkeeping.
    ModelVersion.query.filter_by(target=target, is_current=True).update({'is_current': False})
    last = ModelVersion.query.filter_by(target=target).order_by(
        ModelVersion.version.desc()).first()
    version = (last.version + 1) if last else 1
    mv = ModelVersion(target=target, version=version, algorithm=best,
                      accuracy=evals[best]['accuracy'],
                      n_training_samples=int(valid.sum()),
                      metrics_json=json.dumps(evals), is_current=True)
    db.session.add(mv)

    for s in TrainingSample.query.filter_by(used_for_training=False).all():
        s.used_for_training = True
    db.session.commit()

    predictor.reload()
    return {'target': target, 'version': version, 'algorithm': best,
            'accuracy': evals[best]['accuracy'], 'n_samples': int(valid.sum())}

In [11]:
def retrain_all(config, db, models_module, predictor):
    results = {}
    for t in TARGET_COLS:
        r = retrain(t, config, db, models_module, predictor)
        if r:
            results[t] = r
    return results